In [82]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
from typing import Literal, Optional, Tuple, Union, Dict, Any, List
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

# ==========================================
# 0. GLOBAL PLOTTING & ENVIRONMENT SETUP
# ==========================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

#pl.Config.set_tbl_rows(-1)             # shows all table rows globably
#pl.Config.set_tbl_cols(-1)             # shows all table columns globably
#pl.Config.set_fmt_str_lengths(100)     # increase character count visible globably
#pl.Config.restore_defaults()           # reset to default configuration

# Get current working directory (e.g., .\notebooks)
cwd = os.getcwd()

# Get parent folder to access (.\Data)
parent = os.path.dirname(cwd)

# Get file path to load into environment
file_path_EPTI_data = os.path.join(parent, "Data", "EPTI_CET_data.parquet")

print(f"Loading dataset from: {file_path_EPTI_data}")
df = pl.read_parquet(file_path_EPTI_data).with_columns(pl.col(["Price_profile", "Price_level", "Region", "Which_Phases_Participate",
                                                                "Municipality", "Group_Phase2", "Control_Price_Phase2"]).cast(pl.Categorical))

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\EPTI_CET_data.parquet


In [83]:
# Check for missing values
def what_percentage_missing(df):
    missing_percentage = (
        df.select((pl.all().is_null().sum() / pl.len()) * 100)
        .transpose(include_header=True, header_name="column", column_names=["pct_missing"])
        .sort("pct_missing", descending=True)
    )

    print("Percentage of missing values in each column:")

    with pl.Config(tbl_rows=-1):
        print(missing_percentage)

what_percentage_missing(df)

Percentage of missing values in each column:
shape: (19, 2)
┌──────────────────────────┬─────────────┐
│ column                   ┆ pct_missing │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Price_profile            ┆ 88.006897   │
│ Price_level              ┆ 88.006897   │
│ Experiment_price_NOK_kWh ┆ 88.006897   │
│ ID                       ┆ 0.0         │
│ From                     ┆ 0.0         │
│ Date                     ┆ 0.0         │
│ Day_Of_Week              ┆ 0.0         │
│ Is_Weekend               ┆ 0.0         │
│ Hour                     ┆ 0.0         │
│ Demand_kWh               ┆ 0.0         │
│ Temperature              ┆ 0.0         │
│ Temperature24            ┆ 0.0         │
│ Temperature48            ┆ 0.0         │
│ Temperature72            ┆ 0.0         │
│ Municipality             ┆ 0.0         │
│ Region                   ┆ 0.0         │
│ Control_Price_Phase2     ┆ 0.0     

In [84]:
# Create a boolean expression checking for "Control"
is_control = pl.col("Group_Phase2").cast(pl.String).str.contains("Control")

# Summary DataFrame with counts and percentages
summary_df = (
    df.with_columns(
        pl.when(is_control)
        .then(pl.lit("Control"))
        .otherwise(pl.lit("Experiment"))
        .alias("Cohort")
    )
    .group_by("Cohort")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(2).alias("pct")
    )
)

print(summary_df)

shape: (2, 3)
┌────────────┬─────────┬───────┐
│ Cohort     ┆ count   ┆ pct   │
│ ---        ┆ ---     ┆ ---   │
│ str        ┆ u32     ┆ f64   │
╞════════════╪═════════╪═══════╡
│ Experiment ┆ 8638752 ┆ 70.93 │
│ Control    ┆ 3541248 ┆ 29.07 │
└────────────┴─────────┴───────┘


In [85]:
# Subset containing Ber_Control, Bo_Control, Control, Os_Control, Trom_Control, Trond_Control
df_control = df.filter(is_control)

# Subset containing everything else (Ber_1, Bo_1, H1, H2, NA, Os_1..6, Trom_1..2, Trond_1)
df_experiment = df.filter(~is_control)

In [5]:
# just the dataframe with 
what_percentage_missing(df_experiment)

Percentage of missing values in each column:
shape: (19, 2)
┌──────────────────────────┬─────────────┐
│ column                   ┆ pct_missing │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Price_profile            ┆ 83.090613   │
│ Price_level              ┆ 83.090613   │
│ Experiment_price_NOK_kWh ┆ 83.090613   │
│ ID                       ┆ 0.0         │
│ From                     ┆ 0.0         │
│ Date                     ┆ 0.0         │
│ Day_Of_Week              ┆ 0.0         │
│ Is_Weekend               ┆ 0.0         │
│ Hour                     ┆ 0.0         │
│ Demand_kWh               ┆ 0.0         │
│ Temperature              ┆ 0.0         │
│ Temperature24            ┆ 0.0         │
│ Temperature48            ┆ 0.0         │
│ Temperature72            ┆ 0.0         │
│ Municipality             ┆ 0.0         │
│ Region                   ┆ 0.0         │
│ Control_Price_Phase2     ┆ 0.0     

In [6]:
# 100% missing - as expected
#what_percentage_missing(df_control)

In [86]:
# How many and what proportion of all days fall specifically on weekends: 2,376,394 Control-hours plus another 9,813 Experiment-hours are from a Weekend date 

total_rows = df.height
total_cols = df.width

stats = (
    df
    .filter(
        (pl.col("Is_Weekend") == 1)
        # & (pl.col("Experiment_price_NOK_kWh").is_not_null())
        # & (pl.col("Price_profile").is_not_null())
        # & (pl.col("Price_level").is_not_null())
    )
    .select(
        Match_Count=pl.len(),
        Control_Null_Count=pl.col("Price_level").is_null().sum(),
    )
    .with_columns(
        Pct_of_Total_DF=(pl.col("Match_Count") / total_rows) * 100,
        Control_Null_Pct_of_Total_DF=(pl.col("Control_Null_Count") / total_rows) * 100,
        Control_Null_Pct_of_Matches=(pl.col("Control_Null_Count") / pl.col("Match_Count")) * 100,
    )
)

print(f"Total rows in df: {total_rows} and Total columns in df: {total_cols}")
print(stats)

Total rows in df: 12180000 and Total columns in df: 19
shape: (1, 5)
┌─────────────┬────────────────────┬─────────────────┬──────────────────────┬──────────────────────┐
│ Match_Count ┆ Control_Null_Count ┆ Pct_of_Total_DF ┆ Control_Null_Pct_of_ ┆ Control_Null_Pct_of_ │
│ ---         ┆ ---                ┆ ---             ┆ Total_DF             ┆ Matches              │
│ u32         ┆ u32                ┆ f64             ┆ ---                  ┆ ---                  │
│             ┆                    ┆                 ┆ f64                  ┆ f64                  │
╞═════════════╪════════════════════╪═════════════════╪══════════════════════╪══════════════════════╡
│ 3364375     ┆ 3354562            ┆ 27.622126       ┆ 27.54156             ┆ 99.708326            │
└─────────────┴────────────────────┴─────────────────┴──────────────────────┴──────────────────────┘


In [87]:
# Very few occurances of Expermiental days - it is possible that some of these are just "bleed" from a Friday night midnight into Saturday
# We can not create an Is_Weekend variant of our proposed model, with these data. Bummer.
weekend_anomalies = (
    df
    .filter((pl.col("Is_Weekend") == 1) & (pl.col("Price_level").is_not_null()))
    .group_by(["Municipality", "Price_profile", "Hour"])
    .len()
    .sort("len", descending=True)
)

with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(weekend_anomalies)

# Optional: Remove the 3,364,375 occurances of Control Weekend + 9,813 experiment occurances bleed into the weekend - so that the Is_Weekend column can be dropped
# Also this will ensure that our Baseline kWh measurement only uses weekday values from control - as those are the only days the Experiment was conducted
df_no_weekends = df.remove(pl.col("Is_Weekend") == 1)
print(f"Total rows in df: {df_no_weekends.height} and Total columns in df: {df_no_weekends.width}")


shape: (33, 4)
┌──────────────┬───────────────┬──────┬──────┐
│ Municipality ┆ Price_profile ┆ Hour ┆ len  │
│ ---          ┆ ---           ┆ ---  ┆ ---  │
│ cat          ┆ cat           ┆ i32  ┆ u32  │
╞══════════════╪═══════════════╪══════╪══════╡
│ Oslo         ┆ P             ┆ 0    ┆ 1076 │
│ Tromsø       ┆ P             ┆ 0    ┆ 1060 │
│ Tromsø       ┆ B             ┆ 0    ┆ 1036 │
│ Bergen       ┆ B             ┆ 0    ┆ 807  │
│ Trondheim    ┆ B             ┆ 0    ┆ 804  │
│ Oslo         ┆ B             ┆ 0    ┆ 648  │
│ Oslo         ┆ A             ┆ 0    ┆ 454  │
│ Oslo         ┆ C             ┆ 0    ┆ 449  │
│ Bodø         ┆ B             ┆ 0    ┆ 442  │
│ Oslo         ┆ P0            ┆ 0    ┆ 436  │
│ Bærum        ┆ P             ┆ 0    ┆ 337  │
│ Bærum        ┆ B             ┆ 0    ┆ 192  │
│ Lillestrøm   ┆ P             ┆ 0    ┆ 161  │
│ Stavanger    ┆ B             ┆ 0    ┆ 156  │
│ Stavanger    ┆ P             ┆ 0    ┆ 156  │
│ Bergen       ┆ P             ┆ 0    ┆ 138  

In [88]:
# Check for duplicate rows
duplicate_rows = df.is_duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 0


In [89]:
# Check data types
print("Data types of each column:")
schema_df = pl.DataFrame({
    "Column": df_no_weekends.columns,
    "Data Type": [str(dtype) for dtype in df.dtypes]})

with pl.Config(tbl_rows=-1):
    print(schema_df)

Data types of each column:
shape: (19, 2)
┌──────────────────────────┬─────────────────────────────────┐
│ Column                   ┆ Data Type                       │
│ ---                      ┆ ---                             │
│ str                      ┆ str                             │
╞══════════════════════════╪═════════════════════════════════╡
│ ID                       ┆ String                          │
│ From                     ┆ Datetime(time_unit='ms', time_… │
│ Date                     ┆ Date                            │
│ Day_Of_Week              ┆ Int32                           │
│ Is_Weekend               ┆ Int32                           │
│ Hour                     ┆ Int32                           │
│ Demand_kWh               ┆ Float64                         │
│ Price_profile            ┆ Categorical                     │
│ Price_level              ┆ Categorical                     │
│ Experiment_price_NOK_kWh ┆ Float64                         │
│ Temperature

In [90]:
# Audit and filter raw records following Hofmann & Lindberg's quality protocol:
# - Remove negative values, extreme spikes (>25 kWh), and anomalous households (mean > 9.9 kWh).
# - Identify and flag multi-day consecutive zero-kWh runs (vacant homes / meter dropouts).

print("Data Quality Audit")
#initial_row_count = df.height
initial_row_count = df_no_weekends.height
print(f"Starting with {initial_row_count} rows")
initial_households = df.select("ID").n_unique()

# Filter out invalid individual hourly readings (negative values, extreme spikes > 25 kWh)
df_audited = df_no_weekends.filter(
    (pl.col("Demand_kWh") >= 0.0) & 
    (pl.col("Demand_kWh") <= 25.0)
)
rows_after_hourly_filter = df_audited.height
print(f"Removed {initial_row_count - rows_after_hourly_filter} invalid hourly readings (negative or >25 kWh).")

# Exclude households whose overall mean consumption exceeds 9.9 kWh
household_means = (
    df_audited
    .group_by("ID")
    .agg(pl.col("Demand_kWh").mean().alias("Mean_Demand"))
)

valid_households = (
    household_means
    .filter(pl.col("Mean_Demand") <= 9.9)
    .select("ID")
)

df_audited = df_audited.join(valid_households, on="ID", how="inner")
rows_after_household_filter = df_audited.height
excluded_households = initial_households - df_audited.select("ID").n_unique()
print(f"Excluded {excluded_households} households with mean consumption > 9.9 kWh.")
print(f"Rows remaining after household-level filter: {rows_after_household_filter}")


print("Running Robust Consecutive Zero Audit")

# Ensure data is strictly sorted by ID and chronological Timestamp/Date-Hour
df_audited = df_audited.sort(["ID", "Date", "Hour"])

# Flag rows where demand is exactly zero
df_audited = df_audited.with_columns(
    Is_Zero=(pl.col("Demand_kWh") == 0.0)
)

print("test of audited")
#print(df_audited.head())
print(df_audited.select(pl.col("Is_Zero").not_().sum().alias("false_count")))
#print(df_audited)
# Only 668 instances found where Demand is 0 kWh 0.006% of total rows - we will keep these in the dataset (could be solar panels usage or circut breaker tripping)
# Optional removal
#df_audited = df_audited.remove(pl.col("Is_Zero") == True)

# View shape of dataframe
print(f"Total rows in df: {df_audited.height} and Total columns in df: {df_audited.width}")

Data Quality Audit
Starting with 8815625 rows
Removed 0 invalid hourly readings (negative or >25 kWh).
Excluded 0 households with mean consumption > 9.9 kWh.
Rows remaining after household-level filter: 8815625
Running Robust Consecutive Zero Audit
test of audited
shape: (1, 1)
┌─────────────┐
│ false_count │
│ ---         │
│ u32         │
╞═════════════╡
│ 8814957     │
└─────────────┘
Total rows in df: 8815625 and Total columns in df: 20


In [91]:
# Prepare to calculate hourly FPI (240-hour / 10-day prior lookback window)
epsilon = 1e-6  # Smoothing constant to prevent division-by-zero
window_hours = 240  # 10 days * 24 hours
High_Flexibility_Percentage_Threshold = 0.05  # Establish 5% positive change in FPI as customers who are very responsive to Price Signals

# 1) SORT CHRONOLOGICALLY BY HOUSEHOLD & TIMESTAMP (KEEP ALL COLUMNS INTACT)
df_hourly_sorted = (
    df_audited
    .sort(["ID", "From", "Date", "Hour"])
    .with_columns(
        pl.col("Demand_kWh").alias("E_actual_peak")
    )
)

# 2) CALCULATE E_base_peak FROM UP TO 240 MOST RECENT PRIOR HOURS (NO LEAKAGE)
# Shift by 1 hour within each household (`ID`) so hour `t` only uses prior hours (< t).
df_hourly_baseline = (
    df_hourly_sorted
    .with_columns(
        E_base_peak=(
            pl.col("E_actual_peak")
            .shift(1)   # Move values down by 1 row to ensure we don't use current or future data and prevent leakage
            .rolling_mean(window_size=window_hours, min_samples=1)
            .over("ID")
        ),
        Prior_240h_Mean_Temp=(
            pl.col("Temperature")
            .shift(1)
            .rolling_mean(window_size=window_hours, min_samples=1)
            .over("ID")
        ),
        Prior_240h_Obs_Count=(
            pl.col("E_actual_peak")
            .shift(1)
            .is_not_null()
            .cast(pl.Int32)
            .rolling_sum(window_size=window_hours, min_samples=1)
            .over("ID")
            .fill_null(0)
        ),
    )
)

# 3) FILTER TO HOUSEHOLD-EXPERIMENT-HOURS & COMPUTE HOURLY FPI TARGETS
# Active experiment hours have a non-null Experiment_price_NOK_kWh and at least 1 prior hour baseline.
df_fpi_hourly_experiment = (
    df_hourly_baseline
    .filter(
        pl.col("Experiment_price_NOK_kWh").is_not_null() &
        pl.col("E_base_peak").is_not_null()
    )
    .with_columns(
        Flexibility_Performance_Index=(
            (pl.col("E_base_peak") - pl.col("E_actual_peak")) /
            (pl.col("E_base_peak") + epsilon)
        )
    )
    .with_columns(
        High_Flexibility_Flag=(pl.col("Flexibility_Performance_Index") > High_Flexibility_Percentage_Threshold).cast(pl.Int32)
    )
)

print(f"Hourly rows (all): {df_hourly_baseline.height:,} | Columns: {df_hourly_baseline.width}")
print(f"Household-experiment-hour rows (with FPI): {df_fpi_hourly_experiment.height:,} | Columns: {df_fpi_hourly_experiment.width}")
print(
    df_fpi_hourly_experiment.select([
        "ID", "Date", "Hour", "E_base_peak", "E_actual_peak",
        "Temperature", "Flexibility_Performance_Index", "High_Flexibility_Flag"
    ]).head(10)
)

Hourly rows (all): 8,815,625 | Columns: 24
Household-experiment-hour rows (with FPI): 1,450,947 | Columns: 26
shape: (10, 8)
┌───────┬────────────┬──────┬─────────────┬──────────────┬─────────────┬─────────────┬─────────────┐
│ ID    ┆ Date       ┆ Hour ┆ E_base_peak ┆ E_actual_pea ┆ Temperature ┆ Flexibility ┆ High_Flexib │
│ ---   ┆ ---        ┆ ---  ┆ ---         ┆ k            ┆ ---         ┆ _Performanc ┆ ility_Flag  │
│ str   ┆ date       ┆ i32  ┆ f64         ┆ ---          ┆ f64         ┆ e_Index     ┆ ---         │
│       ┆            ┆      ┆             ┆ f64          ┆             ┆ ---         ┆ i32         │
│       ┆            ┆      ┆             ┆              ┆             ┆ f64         ┆             │
╞═══════╪════════════╪══════╪═════════════╪══════════════╪═════════════╪═════════════╪═════════════╡
│ Exp_1 ┆ 2020-12-16 ┆ 1    ┆ 2.4423625   ┆ 1.945        ┆ 7.1         ┆ 0.20364     ┆ 1           │
│ Exp_1 ┆ 2020-12-16 ┆ 2    ┆ 2.43955     ┆ 2.496        ┆ 7.4     

In [92]:
# Prepare to calculate FPI
epsilon = 1e-6  # Smoothing constant to prevent division-by-zero

# 1) AGGREGATE HOURLY OBSERVATIONS TO HOUSEHOLD-DAY LEVEL (KEEP ALL COLUMNS)
# Note: Adjust non-aggregated columns if any extra columns were added to df_audited
meta_cols = [
    c for c in df_audited.columns
    if c not in {
        "ID", "Date", "Hour", "From", "Demand_kWh",
        "Experiment_price_NOK_kWh", "Temperature",
        "Temperature24", "Temperature48", "Temperature72",
        "Temperature_Bin", "Is_Zero"
    }
]

df_daily = (
    df_audited
    .sort(["ID", "Date", "Hour"])
    .group_by(["ID", "Date"], maintain_order=True)
    .agg(
        [
            pl.col("From").first().alias("From"),
            pl.col("Demand_kWh").sum().alias("E_actual_peak"),
            pl.col("Experiment_price_NOK_kWh").sum().alias("Experiment_price_NOK_kWh"),
            pl.col("Temperature").mean().alias("Temperature"),
            pl.col("Temperature24").mean().alias("Temperature24"),
            pl.col("Temperature48").mean().alias("Temperature48"),
            pl.col("Temperature72").mean().alias("Temperature72"),
            # Count how many hours on this date had an active experiment price
            pl.col("Experiment_price_NOK_kWh").is_not_null().sum().alias("Active_Exp_Hours"),
        ]
        + [pl.col(c).first().alias(c) for c in meta_cols]
    )
    .with_columns(
        # Recalculate 5°C temperature bin from daily mean Temperature
        Temperature_Bin=((pl.col("Temperature") / 5.0).round() * 5.0).alias("Temperature_Bin")
    )
    .sort(["ID", "Date"])
)


# 2) CALCULATE E_base_peak FROM UP TO 10 MOST RECENT PRIOR DAYS (NO LEAKAGE)
# Shift by 1 day within each household so day `d` only uses prior days (< d).
# Using rolling_mean over a 10-row window after shift(1) gives the mean of up to
# 10 immediately preceding observations for that household.
df_daily_baseline = (
    df_daily
    .sort(["ID", "Date"])
    .with_columns(
        E_base_peak=(
            pl.col("E_actual_peak")
            .shift(1)
            .rolling_mean(window_size=10, min_samples=1)
            .over("ID")
        ),
        Prior_10d_Mean_Temp=(
            pl.col("Temperature")
            .shift(1)
            .rolling_mean(window_size=10, min_samples=1)
            .over("ID")
        ),
        Prior_10d_Obs_Count=(
            pl.col("E_actual_peak")
            .shift(1)
            .is_not_null()
            .cast(pl.Int32)
            .rolling_sum(window_size=10, min_samples=1)
            .over("ID")
            .fill_null(0)
        ),
    )
)


# 3) FILTER TO HOUSEHOLD-EXPERIMENT-DAYS & COMPUTE FPI TARGETS
# Household-experiment-days: days with active experiment pricing (Active_Exp_Hours > 0)
# and at least 1 prior day available for E_base_peak.
df_fpi_daily_experiment = (
    df_daily_baseline
    .filter(
        (pl.col("Active_Exp_Hours") > 0) &
        (pl.col("E_base_peak").is_not_null())
    )
    .with_columns(
        Flexibility_Performance_Index=(
            (pl.col("E_base_peak") - pl.col("E_actual_peak")) /
            (pl.col("E_base_peak") + epsilon)
        )
    )
    .with_columns(
        High_Flexibility_Flag=(pl.col("Flexibility_Performance_Index") > High_Flexibility_Percentage_Threshold).cast(pl.Int32)
    )
)

print(f"Daily rows (all): {df_daily_baseline.height:,} | Columns: {df_daily_baseline.width}")
print(f"Household-experiment-day rows (with FPI): {df_fpi_daily_experiment.height:,} | Columns: {df_fpi_daily_experiment.width}")
print(
    df_fpi_daily_experiment.select([
        "ID", "Date", "E_base_peak", "E_actual_peak",
        "Temperature", "Temperature_Bin",
        "Flexibility_Performance_Index", "High_Flexibility_Flag"
    ]).head(10)
)

Daily rows (all): 367,500 | Columns: 23
Household-experiment-day rows (with FPI): 95,982 | Columns: 25
shape: (10, 8)
┌───────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ ID    ┆ Date       ┆ E_base_pea ┆ E_actual_p ┆ Temperatur ┆ Temperatur ┆ Flexibilit ┆ High_Flexi │
│ ---   ┆ ---        ┆ k          ┆ eak        ┆ e          ┆ e_Bin      ┆ y_Performa ┆ bility_Fla │
│ str   ┆ date       ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ nce_Index  ┆ g          │
│       ┆            ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ ---        ┆ ---        │
│       ┆            ┆            ┆            ┆            ┆            ┆ f64        ┆ i32        │
╞═══════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╡
│ Exp_1 ┆ 2020-12-16 ┆ 58.6255    ┆ 55.653     ┆ 7.733333   ┆ 10.0       ┆ 0.050703   ┆ 1          │
│ Exp_1 ┆ 2020-12-17 ┆ 58.4279    ┆ 57.579     ┆ 8.279167   ┆ 10.0       ┆

In [24]:
#what_percentage_missing(df_fpi_hourly_experiment)

In [93]:
with pl.Config(tbl_cols=-1, tbl_width_chars=-1):
    display(df_fpi_hourly_experiment.head())

ID,From,Date,Day_Of_Week,Is_Weekend,Hour,Demand_kWh,Price_profile,Price_level,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72,Municipality,Region,Control_Price_Phase2,Group_Phase2,Which_Phases_Participate,Is_Zero,E_actual_peak,E_base_peak,Prior_240h_Mean_Temp,Prior_240h_Obs_Count,Flexibility_Performance_Index,High_Flexibility_Flag
str,"datetime[ms, UTC]",date,i32,i32,i32,f64,cat,cat,f64,f64,f64,f64,f64,cat,cat,cat,cat,cat,bool,f64,f64,f64,i32,f64,i32
"""Exp_1""",2020-12-16 00:00:00 UTC,2020-12-16,4,0,1,1.945,"""B""","""10""",1.0,7.1,7.6,8.0,6.2,"""Bergen""","""Bergen""","""Price group""","""H1""","""Both""",false,1.945,2.4423625,5.971667,240,0.20364,1
"""Exp_1""",2020-12-16 01:00:00 UTC,2020-12-16,4,0,2,2.496,"""B""","""10""",0.99,7.4,7.6,7.9,6.3,"""Bergen""","""Bergen""","""Price group""","""H1""","""Both""",false,2.496,2.43955,5.976667,240,-0.02314,0
"""Exp_1""",2020-12-16 02:00:00 UTC,2020-12-16,4,0,3,1.866,"""B""","""10""",0.95,6.9,7.6,7.9,6.4,"""Bergen""","""Bergen""","""Price group""","""H1""","""Both""",false,1.866,2.441329,5.982083,240,0.235662,1
"""Exp_1""",2020-12-16 03:00:00 UTC,2020-12-16,4,0,4,1.875,"""B""","""10""",0.95,7.2,7.5,7.9,6.5,"""Bergen""","""Bergen""","""Price group""","""H1""","""Both""",false,1.875,2.43775,5.985417,240,0.230848,1
"""Exp_1""",2020-12-16 04:00:00 UTC,2020-12-16,4,0,5,2.46,"""B""","""10""",0.96,7.4,7.5,7.8,6.6,"""Bergen""","""Bergen""","""Price group""","""H1""","""Both""",false,2.46,2.436883,5.989167,240,-0.009486,0


In [94]:
# drop columns that are not useful or detrimental to our machine learning models
# Team Omega will test with E_base_peak not in the dataset, and with it avaialble - and measure for noise
cols_to_drop = ["Is_Weekend", 
                "Is_Zero", 
                "E_actual_peak", 
                "Prior_240h_Obs_Count", 
                "Demand_kWh", 
                "E_base_peak",
                "Region",
]
df_fpi_hourly_experiment_columns_removed = df_fpi_hourly_experiment.drop(cols_to_drop)#, strict=False) # strict=False lets us remove future columns without re-running the whole notebook
with pl.Config(tbl_cols=-1, tbl_width_chars=-1):
    print(f"Rows (all): {df_fpi_hourly_experiment_columns_removed.height:,} | Columns: {df_fpi_hourly_experiment_columns_removed.width}")
    display(df_fpi_hourly_experiment_columns_removed.head())

Rows (all): 1,450,947 | Columns: 19


ID,From,Date,Day_Of_Week,Hour,Price_profile,Price_level,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72,Municipality,Control_Price_Phase2,Group_Phase2,Which_Phases_Participate,Prior_240h_Mean_Temp,Flexibility_Performance_Index,High_Flexibility_Flag
str,"datetime[ms, UTC]",date,i32,i32,cat,cat,f64,f64,f64,f64,f64,cat,cat,cat,cat,f64,f64,i32
"""Exp_1""",2020-12-16 00:00:00 UTC,2020-12-16,4,1,"""B""","""10""",1.0,7.1,7.6,8.0,6.2,"""Bergen""","""Price group""","""H1""","""Both""",5.971667,0.20364,1
"""Exp_1""",2020-12-16 01:00:00 UTC,2020-12-16,4,2,"""B""","""10""",0.99,7.4,7.6,7.9,6.3,"""Bergen""","""Price group""","""H1""","""Both""",5.976667,-0.02314,0
"""Exp_1""",2020-12-16 02:00:00 UTC,2020-12-16,4,3,"""B""","""10""",0.95,6.9,7.6,7.9,6.4,"""Bergen""","""Price group""","""H1""","""Both""",5.982083,0.235662,1
"""Exp_1""",2020-12-16 03:00:00 UTC,2020-12-16,4,4,"""B""","""10""",0.95,7.2,7.5,7.9,6.5,"""Bergen""","""Price group""","""H1""","""Both""",5.985417,0.230848,1
"""Exp_1""",2020-12-16 04:00:00 UTC,2020-12-16,4,5,"""B""","""10""",0.96,7.4,7.5,7.8,6.6,"""Bergen""","""Price group""","""H1""","""Both""",5.989167,-0.009486,0


In [112]:
def _is_categorical_or_string(dtype: pl.DataType) -> bool:
    """Safely detect parameterized Polars Categorical dtypes as well as String/Utf8."""
    return isinstance(dtype, pl.Categorical) or dtype in (pl.String, pl.Utf8)


def _is_numeric(dtype: pl.DataType) -> bool:
    """Detect standard numeric Polars dtypes (excluding booleans)."""
    return isinstance(
        dtype,
        (
            pl.Int8, pl.Int16, pl.Int32, pl.Int64,
            pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
            pl.Float32, pl.Float64,
        ),
    )


def preprocess_and_split_polars(
    df: pl.DataFrame,
    robust_scaler_cols: Optional[list[str]] = None,
    ohe_cols: Optional[list[str]] = None,
    ignore_cols: Optional[list[str]] = None,
    remove_cols: Optional[list[str]] = None,
    mode: Literal["supervised", "unsupervised"] = "supervised",
    split_pcts: list[int] = [70, 15, 15],
    split_mode: Literal["time", "random", "group"] = "time",
    group_col: Optional[str] = "ID",
    scaling_mode: Literal["robust", "none", "keep_raw"] = "robust",
    ohe_mode: Literal["encode", "none"] = "encode",
    time_cols: list[str] = ["From", "Date", "Hour"],
    random_state: int = 42,
    drop_ohe_original: bool = True,
    keep_row_id: bool = True,
    return_artifacts: bool = False,
) -> Union[
    pl.DataFrame,
    Tuple[pl.DataFrame, pl.DataFrame],
    Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame],
    Tuple[
        Union[pl.DataFrame, Tuple[pl.DataFrame, pl.DataFrame], Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]],
        Dict[str, Any],
    ],
]:
    """
    Preprocess a Polars DataFrame with RobustScaler + OneHotEncoder and either:
      - mode="supervised"   : Split into 2 (train/test) or 3 (train/val/test) splits
                            and fit transformers strictly on the training split.
      - mode="unsupervised" : Keep 100% of rows (no split) for unsupervised models
                            such as K-Means clustering, PCA, HDBSCAN, GMM, etc.

    Parameters
    ----------
    df : pl.DataFrame
        Input Polars DataFrame (e.g., df_fpi_hourly_experiment).
    robust_scaler_cols : list[str] | None
        Columns to scale with RobustScaler. If None, inferred from numeric dtypes
        (excluding ignore_cols, remove_cols, and ohe_cols).
    ohe_cols : list[str] | None
        Columns to one-hot encode. If None, inferred from Categorical/String dtypes
        (excluding ignore_cols and remove_cols).
    ignore_cols : list[str] | None
        Columns to leave untouched in the returned splits (e.g., IDs, timestamps, targets, raw flags).
    remove_cols : list[str] | None
        Columns to remove from the modeling splits prior to preprocessing/splitting,
        while capturing them per split (keyed by `_row_id`) in `artifacts["removed_cols_by_split"]`
        so they can be re-connected later via `.join(..., on="_row_id", how="left")`.
    mode : {"supervised", "unsupervised"}
        "supervised" splits data and fits transformers on train only.
        "unsupervised" returns the full 100% preprocessed DataFrame without splitting.
    split_pcts : list[int]
        Integer percentages summing to 100 (used when mode="supervised").
        Length 2 -> (train, test); Length 3 -> (train, val, test). Default is [70, 15, 15].
    split_mode : {"time", "random", "group"}
        "time" sorts chronologically before slicing.
        "random" shuffles rows reproducibly using `random_state`.
        "group" splits by `group_col` (e.g., "ID") using GroupShuffleSplit so groups
        do not leak across train/val/test.
    group_col : str | None
        Column name used for household/group-aware splitting when split_mode="group".
    time_cols : list[str]
        Columns used for chronological sorting when split_mode="time".
    random_state : int
        Seed for reproducible shuffling/group splitting.
    drop_ohe_original : bool
        If True, replaces original ohe_cols with encoded indicator columns.
    keep_row_id : bool
        If True, keeps `_row_id` on returned split DataFrames so removed columns (or
        original rows) can be joined back on `_row_id`. If False, drops `_row_id`
        from the returned splits.
    return_artifacts : bool
        If True, also returns a dict with the fitted preprocessor, column lists,
        and captured `removed_cols_by_split`.
    scaling_mode options:
      - "robust"   : Fit RobustScaler on train (or full df in unsupervised mode) and replace numeric cols.
      - "none"     : Skip RobustScaler (ideal for tree ensembles like Random Forest / XGBoost).
      - "keep_raw" : Fit RobustScaler, append scaled cols, and keep raw numeric cols.
    ohe_mode options:
      - "encode"   : Fit OneHotEncoder on train (or full df in unsupervised mode) and add OHE columns.
      - "none"     : Skip OneHotEncoder and keep categorical columns as-is (ideal for tree/GB models).
    """
    # --- 1. Validate mode, split_pcts, scaling_mode, ohe_mode & column args ---
    if mode not in ("supervised", "unsupervised"):
        raise ValueError("`mode` must be either 'supervised' or 'unsupervised'.")
    if scaling_mode not in ("robust", "none", "keep_raw"):
        raise ValueError("`scaling_mode` must be one of 'robust', 'none', or 'keep_raw'.")
    if ohe_mode not in ("encode", "none"):
        raise ValueError("`ohe_mode` must be either 'encode' or 'none'.")

    if mode == "supervised":
        if not isinstance(split_pcts, list) or len(split_pcts) not in (2, 3):
            raise ValueError("`split_pcts` must be a list of 2 integers [train, test] or 3 integers [train, val, test].")
        if not all(isinstance(p, int) and not isinstance(p, bool) and p > 0 for p in split_pcts):
            raise ValueError("All elements in `split_pcts` must be positive integers.")
        if sum(split_pcts) != 100:
            raise ValueError(f"`split_pcts` must sum to 100, got {sum(split_pcts)} ({split_pcts}).")

    row_id_col = "_row_id"
    if row_id_col in df.columns:
        raise ValueError(f"Input DataFrame already contains reserved column '{row_id_col}'.")

    ignored = set(ignore_cols or [])
    missing_ignored = ignored - set(df.columns)
    if missing_ignored:
        raise ValueError(f"`ignore_cols` contains columns not in DataFrame: {sorted(missing_ignored)}")

    removed_list = list(dict.fromkeys(remove_cols or []))
    missing_removed = set(removed_list) - set(df.columns)
    if missing_removed:
        raise ValueError(f"`remove_cols` contains columns not in DataFrame: {sorted(missing_removed)}")

    # Prevent conflicting instructions on the same column
    overlap_ignore_remove = ignored & set(removed_list)
    if overlap_ignore_remove:
        raise ValueError(f"Columns cannot be in both `ignore_cols` and `remove_cols`: {sorted(overlap_ignore_remove)}")

    removed_set = set(removed_list)
    excluded_from_features = ignored | removed_set

    # Attach a deterministic row ID prior to ordering/splitting so removed columns can be re-linked later
    df_indexed = df.with_row_index(name=row_id_col)

    # --- 2. Infer categorical & numeric feature columns ---
    if ohe_cols is None:
        inferred_ohe_cols = [
            col for col, dtype in df.schema.items()
            if col not in excluded_from_features and _is_categorical_or_string(dtype)
        ]
        print(f'Infered OHE: {inferred_ohe_cols}')
    else:
        missing_ohe = set(ohe_cols) - set(df.columns)
        if missing_ohe:
            raise ValueError(f"`ohe_cols` contains columns not in DataFrame: {sorted(missing_ohe)}")
        inferred_ohe_cols = [c for c in ohe_cols if c not in excluded_from_features]

    overlap = set(inferred_ohe_cols) & (set(robust_scaler_cols or []))
    if overlap:
        raise ValueError(f"Columns cannot be in both `ohe_cols` and `robust_scaler_cols`: {sorted(overlap)}")

    if robust_scaler_cols is None:
        inferred_num_cols = [
            col for col, dtype in df.schema.items()
            if col not in excluded_from_features and col not in inferred_ohe_cols and _is_numeric(dtype)
        ]
        print(f'Infered Num: {inferred_num_cols}')
    else:
        missing_num = set(robust_scaler_cols) - set(df.columns)
        if missing_num:
            raise ValueError(f"`robust_scaler_cols` contains columns not in DataFrame: {sorted(missing_num)}")
        inferred_num_cols = [c for c in robust_scaler_cols if c not in excluded_from_features and c not in inferred_ohe_cols]

    active_num_cols = inferred_num_cols if scaling_mode in ("robust", "keep_raw") else []
    active_ohe_cols = inferred_ohe_cols if ohe_mode == "encode" else []

    # --- 3. Order & split data (or keep 100% in unsupervised mode) ---
    if mode == "unsupervised":
        sort_by = [c for c in time_cols if c in df_indexed.columns]
        raw_splits = [df_indexed.sort(sort_by) if sort_by else df_indexed]
        fit_source_df = raw_splits[0]
        split_names = ["full"]
    else:
        if split_mode == "time":
            sort_by = [c for c in time_cols if c in df_indexed.columns]
            working_df = df_indexed.sort(sort_by) if sort_by else df_indexed
            n_rows = working_df.height
            if n_rows < len(split_pcts):
                raise ValueError("DataFrame has fewer rows than requested splits.")

            if len(split_pcts) == 3:
                train_pct, val_pct, _ = split_pcts
                n_train = int(n_rows * (train_pct / 100.0))
                n_val = int(n_rows * (val_pct / 100.0))
                raw_splits = [
                    working_df.slice(0, n_train),
                    working_df.slice(n_train, n_val),
                    working_df.slice(n_train + n_val, n_rows - n_train - n_val),
                ]
                split_names = ["train", "val", "test"]
            else:
                train_pct, _ = split_pcts
                n_train = int(n_rows * (train_pct / 100.0))
                raw_splits = [
                    working_df.slice(0, n_train),
                    working_df.slice(n_train, n_rows - n_train),
                ]
                split_names = ["train", "test"]

        elif split_mode == "random":
            working_df = df_indexed.sample(fraction=1.0, shuffle=True, seed=random_state)
            n_rows = working_df.height
            if n_rows < len(split_pcts):
                raise ValueError("DataFrame has fewer rows than requested splits.")

            if len(split_pcts) == 3:
                train_pct, val_pct, _ = split_pcts
                n_train = int(n_rows * (train_pct / 100.0))
                n_val = int(n_rows * (val_pct / 100.0))
                raw_splits = [
                    working_df.slice(0, n_train),
                    working_df.slice(n_train, n_val),
                    working_df.slice(n_train + n_val, n_rows - n_train - n_val),
                ]
                split_names = ["train", "val", "test"]
            else:
                train_pct, _ = split_pcts
                n_train = int(n_rows * (train_pct / 100.0))
                raw_splits = [
                    working_df.slice(0, n_train),
                    working_df.slice(n_train, n_rows - n_train),
                ]
                split_names = ["train", "test"]

        elif split_mode == "group":
            if not group_col or group_col not in df_indexed.columns:
                raise ValueError(f"`group_col` ('{group_col}') must be a valid column in DataFrame when split_mode='group'.")

            row_idx = np.arange(df_indexed.height)
            groups = df_indexed.get_column(group_col).to_numpy()

            if len(split_pcts) == 3:
                train_pct, val_pct, test_pct = split_pcts
                test_frac = test_pct / 100.0
                val_frac_of_remaining = val_pct / (train_pct + val_pct)

                gss_test = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=random_state)
                train_val_idx, test_idx = next(gss_test.split(row_idx, groups=groups))

                gss_val = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_remaining, random_state=random_state)
                train_rel_idx, val_rel_idx = next(gss_val.split(train_val_idx, groups=groups[train_val_idx]))
                train_idx = train_val_idx[train_rel_idx]
                val_idx = train_val_idx[val_rel_idx]

                raw_splits = [
                    df_indexed[train_idx.tolist()],
                    df_indexed[val_idx.tolist()],
                    df_indexed[test_idx.tolist()],
                ]
                split_names = ["train", "val", "test"]
            else:
                train_pct, test_pct = split_pcts
                gss = GroupShuffleSplit(n_splits=1, test_size=test_pct / 100.0, random_state=random_state)
                train_idx, test_idx = next(gss.split(row_idx, groups=groups))
                raw_splits = [
                    df_indexed[train_idx.tolist()],
                    df_indexed[test_idx.tolist()],
                ]
                split_names = ["train", "test"]
        else:
            raise ValueError("`split_mode` must be 'time', 'random', or 'group'.")

        # Fit strictly on training split to prevent leakage
        fit_source_df = raw_splits[0]

    # --- 4. Capture remove_cols per split (with _row_id) and drop from splits ---
    removed_cols_by_split: Dict[str, pl.DataFrame] = {}
    splits_without_removed: list[pl.DataFrame] = []

    for split_name, split_df in zip(split_names, raw_splits):
        if removed_list:
            removed_cols_by_split[split_name] = split_df.select([row_id_col] + removed_list)
            splits_without_removed.append(split_df.drop(removed_list))
        else:
            # Still store _row_id mapping so callers can always look up row order/IDs if desired
            removed_cols_by_split[split_name] = split_df.select([row_id_col])
            splits_without_removed.append(split_df)

    fit_source_df = splits_without_removed[0]

    # --- 5. Fit transformers on training split (or full df in unsupervised mode) ---
    cols_to_fit = active_num_cols + active_ohe_cols
    print(f'Columns Transformed: {cols_to_fit}')
    transformed_splits: list[pl.DataFrame] = []
    preprocessor: Optional[ColumnTransformer] = None
    encoded_col_names: list[str] = []

    if cols_to_fit:
        transformers = []
        if active_num_cols:
            transformers.append(("num", RobustScaler(), active_num_cols))
        if active_ohe_cols:
            transformers.append((
                "cat",
                OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop=None),  # drop options {'first', 'if_binary'} for unregularized linear/logistic regression
                active_ohe_cols,
            ))

        preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

        fit_pd = fit_source_df.select(cols_to_fit).to_pandas()
        for c in active_ohe_cols:
            fit_pd[c] = fit_pd[c].astype(str)

        preprocessor.fit(fit_pd)
        encoded_col_names = preprocessor.get_feature_names_out().tolist()

        for split_df in splits_without_removed:
            split_pd = split_df.select(cols_to_fit).to_pandas()
            for c in active_ohe_cols:
                split_pd[c] = split_pd[c].astype(str)

            arr = preprocessor.transform(split_pd)
            if hasattr(arr, "toarray"):
                arr = arr.toarray()
            elif hasattr(arr, "to_numpy"):
                arr = arr.to_numpy()

            transformed_pl = pl.DataFrame({
                name: arr[:, idx] for idx, name in enumerate(encoded_col_names)
            })

            cols_to_drop_from_base = []
            if drop_ohe_original and active_ohe_cols:
                cols_to_drop_from_base.extend(active_ohe_cols)
            if scaling_mode == "robust" and active_num_cols:
                cols_to_drop_from_base.extend(active_num_cols)

            base_pl = split_df.drop(cols_to_drop_from_base) if cols_to_drop_from_base else split_df
            merged_pl = pl.concat([base_pl, transformed_pl], how="horizontal")
            if not keep_row_id:
                merged_pl = merged_pl.drop(row_id_col)
            transformed_splits.append(merged_pl)
    else:
        for split_df in splits_without_removed:
            transformed_splits.append(split_df if keep_row_id else split_df.drop(row_id_col))

    result_payload: Union[
        pl.DataFrame,
        Tuple[pl.DataFrame, pl.DataFrame],
        Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame],
    ] = transformed_splits[0] if mode == "unsupervised" else tuple(transformed_splits)

    if return_artifacts:
        artifacts = {
            "preprocessor": preprocessor,
            "detected_num_cols": inferred_num_cols,
            "scaled_num_cols": active_num_cols,
            "ohe_cols": inferred_ohe_cols,
            "encoded_ohe_cols": active_ohe_cols,
            "encoded_feature_names": encoded_col_names,
            "ignored_cols": sorted(ignored),
            "removed_cols": removed_list,
            "removed_cols_by_split": removed_cols_by_split,
            "row_id_col": row_id_col if keep_row_id else None,
            "scaling_mode": scaling_mode,
            "ohe_mode": ohe_mode,
            "mode": mode,
            "split_mode": None if mode == "unsupervised" else split_mode,
            "group_col": group_col if split_mode == "group" else None,
            "split_names": split_names,
        }
        return result_payload, artifacts

    return result_payload



""" # Reconnect removed columns (e.g., ID, From, Date, targets) to test predictions/results
test_with_removed = df_test.join(
    artifacts["removed_cols_by_split"]["test"],
    on="_row_id",
    how="left",
)

# Or reconnect inside a GroupKFold validation slice using val_idx
val_slice_with_removed = (
    df_train[val_idx.tolist()]
    .join(artifacts["removed_cols_by_split"]["train"], on="_row_id", how="left")
) """

' # Reconnect removed columns (e.g., ID, From, Date, targets) to test predictions/results\ntest_with_removed = df_test.join(\n    artifacts["removed_cols_by_split"]["test"],\n    on="_row_id",\n    how="left",\n)\n\n# Or reconnect inside a GroupKFold validation slice using val_idx\nval_slice_with_removed = (\n    df_train[val_idx.tolist()]\n    .join(artifacts["removed_cols_by_split"]["train"], on="_row_id", how="left")\n) '

In [120]:
ignore_cols_regression = [
    "Flexibility_Performance_Index",
]

remove_cols_regression=[
    "ID",
    "From",
    "Date",
    "High_Flexibility_Flag",
]

# 1) Supervised 80/20 Household-Aware Split For Neural Network or Linear - Numeric Scaled and Categorical One Hot Encoded
(df_train_linear_nn_regression, df_test_linear_nn_regression), artifacts_linear_nn_regression = preprocess_and_split_polars(
    df=df_fpi_hourly_experiment_columns_removed,
    ignore_cols=ignore_cols_regression,
    remove_cols=remove_cols_regression,
    mode="supervised",
    split_pcts=[80, 20],
    split_mode="group",
    group_col="ID",
    scaling_mode="robust",  # Use "none" for Random Forest / XGBoost / GBDT
    random_state=42,
    return_artifacts=True,
)

# 2) Household (ID)-Aware Cross-Validation Inside df_train (5-Fold GroupKFold)
target_col = "Flexibility_Performance_Index"
exclude_from_X = {"_row_id", target_col, *(ignore_cols_regression or remove_cols_regression or [])}
feature_cols = [
    c for c in df_train_linear_nn_regression.columns
    if c not in exclude_from_X
]
X_train_full_linear_nn_regression = df_train_linear_nn_regression.select(feature_cols).to_numpy()
y_train_reg_linear_nn_regression = df_train_linear_nn_regression.get_column(target_col).to_numpy()
groups_train_linear_nn_regression = artifacts_linear_nn_regression["removed_cols_by_split"]["train"].get_column("ID").to_numpy()

gkf = GroupKFold(n_splits=5)
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train_full_linear_nn_regression, y_train_reg_linear_nn_regression, groups=groups_train_linear_nn_regression), start=1):
    X_train_linear_nn_regression, y_train_linear_nn_regression = X_train_full_linear_nn_regression[tr_idx], y_train_reg_linear_nn_regression[tr_idx]
    X_val_linear_nn_regression, y_val_linear_nn_regression = X_train_full_linear_nn_regression[val_idx], y_train_reg_linear_nn_regression[val_idx]
    # Fit your model on (X_tr, y_tr) and evaluate on (X_va, y_va)
    print(f"Linear & Neural Net k-Fold {fold}: train rows={len(tr_idx):,}, val rows={len(val_idx):,}")

# 3) Supervised 80/20 Household-Aware Split For Gradient Boost and Tree based ensembles - No Numeric Scaling or Categorical One Hot Encoding
(df_train_tree_boost_regression, df_test_tree_boost_regression), artifacts_tree_boost_regression = preprocess_and_split_polars(
    df=df_fpi_hourly_experiment_columns_removed,
    ignore_cols=ignore_cols_regression,
    remove_cols=remove_cols_regression,
    mode="supervised",
    split_pcts=[80, 20],
    split_mode="group",
    group_col="ID",
    scaling_mode="none",  # Use "none" for Random Forest / XGBoost / GBDT
    ohe_mode="none",
    random_state=42,
    return_artifacts=True,
)

# 4) Household (ID)-Aware Cross-Validation Inside df_train (5-Fold GroupKFold)
target_col = "Flexibility_Performance_Index"
exclude_from_X = {"_row_id", target_col, *(ignore_cols_regression or remove_cols_regression or [])}
feature_cols = [
    c for c in df_train_tree_boost_regression.columns
    if c not in exclude_from_X
]
X_train_full_tree_boost_regression = df_train_tree_boost_regression.select(feature_cols).to_numpy()
y_train_reg_tree_boost_regression = df_train_tree_boost_regression.get_column(target_col).to_numpy()
groups_train_tree_boost_regression = artifacts_tree_boost_regression["removed_cols_by_split"]["train"].get_column("ID").to_numpy()

gkf = GroupKFold(n_splits=5)
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train_full_tree_boost_regression, y_train_reg_tree_boost_regression, groups=groups_train_tree_boost_regression), start=1):
    X_train_tree_boost_regression, y_train_tree_boost_regression = X_train_full_tree_boost_regression[tr_idx], y_train_reg_tree_boost_regression[tr_idx]
    X_val_tree_boost_regression, y_val_tree_boost_regression = X_train_full_tree_boost_regression[val_idx], y_train_reg_tree_boost_regression[val_idx]
    # Fit your model on (X_tr, y_tr) and evaluate on (X_va, y_va)
    print(f"GB & Tree-Ensemble k-Fold {fold}: train rows={len(tr_idx):,}, val rows={len(val_idx):,}")


remove_cols_unsupervised=[
    "ID",
    "From",
    "Date",
    "Flexibility_Performance_Index",
    "High_Flexibility_Flag",
]

# 5) Unsupervised (100% Data, No Split) for K-Means Clustering
df_unsupervised, artifacts_unsupervised = preprocess_and_split_polars(
    df=df_fpi_hourly_experiment_columns_removed,
    ignore_cols=None,
    remove_cols=remove_cols_unsupervised,
    mode="unsupervised",
    scaling_mode="none",
    ohe_mode="none",
    return_artifacts=True,
)

Infered OHE: ['Price_profile', 'Price_level', 'Municipality', 'Control_Price_Phase2', 'Group_Phase2', 'Which_Phases_Participate']
Infered Num: ['Day_Of_Week', 'Hour', 'Experiment_price_NOK_kWh', 'Temperature', 'Temperature24', 'Temperature48', 'Temperature72', 'Prior_240h_Mean_Temp']
Columns Transformed: ['Day_Of_Week', 'Hour', 'Experiment_price_NOK_kWh', 'Temperature', 'Temperature24', 'Temperature48', 'Temperature72', 'Prior_240h_Mean_Temp', 'Price_profile', 'Price_level', 'Municipality', 'Control_Price_Phase2', 'Group_Phase2', 'Which_Phases_Participate']
Linear & Neural Net k-Fold 1: train rows=928,230, val rows=232,332
Linear & Neural Net k-Fold 2: train rows=928,230, val rows=232,332
Linear & Neural Net k-Fold 3: train rows=928,612, val rows=231,950
Linear & Neural Net k-Fold 4: train rows=928,588, val rows=231,974
Linear & Neural Net k-Fold 5: train rows=928,588, val rows=231,974
Infered OHE: ['Price_profile', 'Price_level', 'Municipality', 'Control_Price_Phase2', 'Group_Phase2',

In [123]:
ignore_cols_classification = [
    "High_Flexibility_Flag",
]

remove_cols_classification=[
    "ID",
    "From",
    "Date",
    "Flexibility_Performance_Index",
]

# 1) CLASSIFICATION Supervised 80/20 Household-Aware Split For Neural Network or Linear - Numeric Scaled and Categorical One Hot Encoded
(df_train_linear_nn_classification, df_test_linear_nn_classification), artifacts_linear_nn_classification = preprocess_and_split_polars(
    df=df_fpi_hourly_experiment_columns_removed,
    ignore_cols=ignore_cols_classification,
    remove_cols=remove_cols_classification,
    mode="supervised",
    split_pcts=[80, 20],
    split_mode="group",
    group_col="ID",
    scaling_mode="robust",  # Use "none" for Random Forest / XGBoost / GBDT
    random_state=42,
    return_artifacts=True,
)

# 2) Household (ID)-Aware Cross-Validation Inside df_train (5-Fold GroupKFold)
target_col = "High_Flexibility_Flag"
exclude_from_X = {"_row_id", target_col, *(ignore_cols_classification or remove_cols_classification or [])}
feature_cols = [
    c for c in df_train_linear_nn_classification.columns
    if c not in exclude_from_X
]
X_train_full_linear_nn_classification = df_train_linear_nn_classification.select(feature_cols).to_numpy()
y_train_reg_linear_nn_classification = df_train_linear_nn_classification.get_column(target_col).to_numpy()
groups_train_linear_nn_classification = artifacts_linear_nn_classification["removed_cols_by_split"]["train"].get_column("ID").to_numpy()

gkf = GroupKFold(n_splits=5)
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train_full_linear_nn_classification, y_train_reg_linear_nn_classification, groups=groups_train_linear_nn_classification), start=1):
    X_train_linear_nn_classification, y_train_linear_nn_classification = X_train_full_linear_nn_classification[tr_idx], y_train_reg_linear_nn_classification[tr_idx]
    X_val_linear_nn_classification, y_val_linear_nn_classification = X_train_full_linear_nn_classification[val_idx], y_train_reg_linear_nn_classification[val_idx]
    # Fit your model on (X_tr, y_tr) and evaluate on (X_va, y_va)
    print(f"Linear & Neural Net k-Fold {fold}: train rows={len(tr_idx):,}, val rows={len(val_idx):,}")

# 3) Supervised 80/20 Household-Aware Split For Gradient Boost and Tree based ensembles - No Numeric Scaling or Categorical One Hot Encoding
(df_train_tree_boost_classification, df_test_tree_boost_classification), artifacts_tree_boost_classification = preprocess_and_split_polars(
    df=df_fpi_hourly_experiment_columns_removed,
    ignore_cols=ignore_cols_classification,
    remove_cols=remove_cols_classification,
    mode="supervised",
    split_pcts=[80, 20],
    split_mode="group",
    group_col="ID",
    scaling_mode="none",  # Use "none" for Random Forest / XGBoost / GBDT
    ohe_mode="none",
    random_state=42,
    return_artifacts=True,
)

# 4) Household (ID)-Aware Cross-Validation Inside df_train (5-Fold GroupKFold)
target_col = "High_Flexibility_Flag"
exclude_from_X = {"_row_id", target_col, *(ignore_cols_classification or remove_cols_classification or [])}
feature_cols = [
    c for c in df_train_tree_boost_classification.columns
    if c not in exclude_from_X
]
X_train_full_tree_boost_classification = df_train_tree_boost_classification.select(feature_cols).to_numpy()
y_train_reg_tree_boost_classification = df_train_tree_boost_classification.get_column(target_col).to_numpy()
groups_train_tree_boost_classification = artifacts_tree_boost_classification["removed_cols_by_split"]["train"].get_column("ID").to_numpy()

gkf = GroupKFold(n_splits=5)
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train_full_tree_boost_classification, y_train_reg_tree_boost_classification, groups=groups_train_tree_boost_classification), start=1):
    X_train_tree_boost_classification, y_train_tree_boost_classification = X_train_full_tree_boost_classification[tr_idx], y_train_reg_tree_boost_classification[tr_idx]
    X_val_tree_boost_classification, y_val_tree_boost_classification = X_train_full_tree_boost_classification[val_idx], y_train_reg_tree_boost_classification[val_idx]
    # Fit your model on (X_tr, y_tr) and evaluate on (X_va, y_va)
    print(f"GB & Tree-Ensemble k-Fold {fold}: train rows={len(tr_idx):,}, val rows={len(val_idx):,}")


Infered OHE: ['Price_profile', 'Price_level', 'Municipality', 'Control_Price_Phase2', 'Group_Phase2', 'Which_Phases_Participate']
Infered Num: ['Day_Of_Week', 'Hour', 'Experiment_price_NOK_kWh', 'Temperature', 'Temperature24', 'Temperature48', 'Temperature72', 'Prior_240h_Mean_Temp']
Columns Transformed: ['Day_Of_Week', 'Hour', 'Experiment_price_NOK_kWh', 'Temperature', 'Temperature24', 'Temperature48', 'Temperature72', 'Prior_240h_Mean_Temp', 'Price_profile', 'Price_level', 'Municipality', 'Control_Price_Phase2', 'Group_Phase2', 'Which_Phases_Participate']
Linear & Neural Net k-Fold 1: train rows=928,230, val rows=232,332
Linear & Neural Net k-Fold 2: train rows=928,230, val rows=232,332
Linear & Neural Net k-Fold 3: train rows=928,612, val rows=231,950
Linear & Neural Net k-Fold 4: train rows=928,588, val rows=231,974
Linear & Neural Net k-Fold 5: train rows=928,588, val rows=231,974
Infered OHE: ['Price_profile', 'Price_level', 'Municipality', 'Control_Price_Phase2', 'Group_Phase2',

In [122]:
with pl.Config(tbl_cols=-1, tbl_width_chars=-1):
    print(f"Rows (all): {df_unsupervised.height:,} | Columns: {df_unsupervised.width}")
    display(df_unsupervised.head())

Rows (all): 1,450,947 | Columns: 15


_row_id,Day_Of_Week,Hour,Price_profile,Price_level,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72,Municipality,Control_Price_Phase2,Group_Phase2,Which_Phases_Participate,Prior_240h_Mean_Temp
u32,i32,i32,cat,cat,f64,f64,f64,f64,f64,cat,cat,cat,cat,f64
346401,4,1,"""P""","""10""",1.0,1.4,2.6,2.6,2.6,"""Oslo""","""Price group""","""Os_2""","""Phase_2""",2.27625
885714,4,1,"""B""","""10""",1.0,1.4,2.6,2.6,2.6,"""Bærum""","""Price group""","""Os_1""","""Phase_2""",2.27625
248145,4,1,"""B""","""10""",1.0,1.4,2.6,2.6,2.6,"""Oslo""","""Price group""","""Os_3""","""Phase_2""",2.27625
486582,4,1,"""A""","""5""",1.0,1.4,2.6,2.6,2.6,"""Oslo""","""Price group""","""Os_6""","""Phase_2""",2.27625
675672,4,1,"""B""","""10""",1.0,7.1,7.6,8.0,6.2,"""Bergen""","""Price group""","""Ber_1""","""Phase_2""",5.971667


In [117]:
with pl.Config(tbl_cols=-1, tbl_width_chars=-1):
    print(f"Rows (all): {df_train_linear_nn_regression.height:,} | Columns: {df_train_linear_nn_regression.width}")
    display(df_train_linear_nn_regression.head())

Rows (all): 1,160,562 | Columns: 47


_row_id,Flexibility_Performance_Index,num__Day_Of_Week,num__Hour,num__Experiment_price_NOK_kWh,num__Temperature,num__Temperature24,num__Temperature48,num__Temperature72,num__Prior_240h_Mean_Temp,cat__Price_profile_A,cat__Price_profile_B,cat__Price_profile_C,cat__Price_profile_P,cat__Price_profile_P0,cat__Price_level_10,cat__Price_level_15,cat__Price_level_2,cat__Price_level_30,cat__Price_level_5,cat__Price_level_5_10,cat__Municipality_Asker,cat__Municipality_Bergen,cat__Municipality_Bodø,cat__Municipality_Bærum,cat__Municipality_Drammen,cat__Municipality_Lillestrøm,cat__Municipality_Oslo,cat__Municipality_Stavanger,cat__Municipality_Tromsø,cat__Municipality_Trondheim,cat__Control_Price_Phase2_Price group,cat__Group_Phase2_Ber_1,cat__Group_Phase2_Bo_1,cat__Group_Phase2_H1,cat__Group_Phase2_H2,cat__Group_Phase2_Os_1,cat__Group_Phase2_Os_2,cat__Group_Phase2_Os_3,cat__Group_Phase2_Os_4,cat__Group_Phase2_Os_5,cat__Group_Phase2_Os_6,cat__Group_Phase2_Trom_1,cat__Group_Phase2_Trom_2,cat__Group_Phase2_Trond_1,cat__Which_Phases_Participate_Both,cat__Which_Phases_Participate_Phase_2
u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
429,0.105795,0.0,-0.916667,-0.024814,0.918033,1.028169,1.026316,1.0,0.961954,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
430,0.10294,0.0,-0.833333,-0.027295,0.852459,1.028169,1.026316,1.0,0.96426,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
431,0.085148,0.0,-0.75,-0.037221,0.770492,1.028169,1.013158,0.987013,0.965485,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
432,0.051048,0.0,-0.666667,-0.037221,0.737705,1.014085,1.013158,0.987013,0.966422,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
433,0.066111,0.0,-0.583333,-0.034739,0.721311,1.0,1.0,0.987013,0.967142,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [118]:
with pl.Config(tbl_cols=-1, tbl_width_chars=-1):
    print(f"Rows (all): {df_train_tree_boost_regression.height:,} | Columns: {df_train_tree_boost_regression.width}")
    display(df_train_tree_boost_regression.head())

Rows (all): 1,160,562 | Columns: 16


_row_id,Day_Of_Week,Hour,Price_profile,Price_level,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72,Municipality,Control_Price_Phase2,Group_Phase2,Which_Phases_Participate,Prior_240h_Mean_Temp,Flexibility_Performance_Index
u32,i32,i32,cat,cat,f64,f64,f64,f64,f64,cat,cat,cat,cat,f64,f64
429,4,1,"""B""","""10""",1.0,1.4,2.6,2.6,2.6,"""Oslo""","""Price group""","""H1""","""Both""",2.27625,0.105795
430,4,2,"""B""","""10""",0.99,1.0,2.6,2.6,2.6,"""Oslo""","""Price group""","""H1""","""Both""",2.289583,0.10294
431,4,3,"""B""","""10""",0.95,0.5,2.6,2.5,2.5,"""Oslo""","""Price group""","""H1""","""Both""",2.296667,0.085148
432,4,4,"""B""","""10""",0.95,0.3,2.5,2.5,2.5,"""Oslo""","""Price group""","""H1""","""Both""",2.302083,0.051048
433,4,5,"""B""","""10""",0.96,0.2,2.4,2.4,2.5,"""Oslo""","""Price group""","""H1""","""Both""",2.30625,0.066111


In [119]:
# drop columns for the Unsupervised dataset that are not useful or detrimental to our machine learning models

cols_to_drop = [
    "_row_id",
]
df_unsupervised_columns_removed = df_unsupervised.drop(cols_to_drop)#, strict=False) # strict=False lets us remove future columns without re-running the whole notebook
with pl.Config(tbl_cols=-1, tbl_width_chars=-1):
    print(f"Rows (all): {df_unsupervised_columns_removed.height:,} | Columns: {df_unsupervised_columns_removed.width}")
    display(df_unsupervised_columns_removed.head())

Rows (all): 1,450,947 | Columns: 14


Day_Of_Week,Hour,Price_profile,Price_level,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72,Municipality,Control_Price_Phase2,Group_Phase2,Which_Phases_Participate,Prior_240h_Mean_Temp
i32,i32,cat,cat,f64,f64,f64,f64,f64,cat,cat,cat,cat,f64
4,1,"""P""","""10""",1.0,1.4,2.6,2.6,2.6,"""Oslo""","""Price group""","""Os_2""","""Phase_2""",2.27625
4,1,"""B""","""10""",1.0,1.4,2.6,2.6,2.6,"""Bærum""","""Price group""","""Os_1""","""Phase_2""",2.27625
4,1,"""B""","""10""",1.0,1.4,2.6,2.6,2.6,"""Oslo""","""Price group""","""Os_3""","""Phase_2""",2.27625
4,1,"""A""","""5""",1.0,1.4,2.6,2.6,2.6,"""Oslo""","""Price group""","""Os_6""","""Phase_2""",2.27625
4,1,"""B""","""10""",1.0,7.1,7.6,8.0,6.2,"""Bergen""","""Price group""","""Ber_1""","""Phase_2""",5.971667
